In [1]:
import glob
import pandas as pd
from collections import Counter
import os
import re

files = glob.glob('/Volumes/Current_Projects2-CC0931-Comparative_Medicine/Lab_Projects/Belief_Updating_Anxiety/Reversal_Data/AX_*/AX_*_day1.csv')

print(f"Found {len(files)} files matching the pattern.")

all_data = []
sub_file_counts = Counter()
excluded_subs = []
manual_excluded_subs = {'2', '26', '40', '50', '53', '78', '79'}
manually_removed = []
kept_subs_trials = {}

for f in sorted(files):
    filename = os.path.basename(f)
    match = re.search(r'AX_(\w+)_day1\.csv', filename)
    if match:
        sub_id = match.group(1)
    else:
        sub_id = filename
    
    sub_file_counts[sub_id] += 1
    
    if sub_id in manual_excluded_subs:
        manually_removed.append(sub_id)
        continue
    
    try:
        df = pd.read_csv(f)
    except Exception as e:
        print(f"Error reading {f}: {e}")
        continue
    
    # 1. Detect reversal (multiple unique values for Pwin)
    w1_unique = df['W1_Pwin'].dropna().unique() if 'W1_Pwin' in df.columns else []
    l1_unique = df['L1_Pwin'].dropna().unique() if 'L1_Pwin' in df.columns else []
    
    if len(w1_unique) > 1 or len(l1_unique) > 1:
        excluded_subs.append(sub_id)
        continue
        
    # 2. Re-map L1/L2 so that L1 is always the better option (since they start < 0.5)
    # Also check W1 just in case, though they are all > 0.5
    
    # Check W1
    if len(w1_unique) == 1 and w1_unique[0] < 0.5:
        # Swap W1 and W2 Pwin
        if 'W2_Pwin' in df.columns:
            temp_w = df['W1_Pwin'].copy()
            df['W1_Pwin'] = df['W2_Pwin']
            df['W2_Pwin'] = temp_w
        # Swap W1/W2 in IDs
        for col in ['LeftID', 'RightID', 'ChosenID']:
            if col in df.columns:
                df[col] = df[col].replace({'W1': 'W2', 'W2': 'W1'})

    # Check L1
    if len(l1_unique) == 1 and l1_unique[0] < 0.5:
        # Swap L1 and L2 Pwin
        if 'L2_Pwin' in df.columns:
            temp_l = df['L1_Pwin'].copy()
            df['L1_Pwin'] = df['L2_Pwin']
            df['L2_Pwin'] = temp_l
        # Swap L1/L2 in IDs
        for col in ['LeftID', 'RightID', 'ChosenID']:
            if col in df.columns:
                df[col] = df[col].replace({'L1': 'L2', 'L2': 'L1'})

    num_trials = len(df)
    kept_subs_trials[sub_id] = kept_subs_trials.get(sub_id, 0) + num_trials
                
    cols_to_keep = ['Sub', 'TrialType', 'IdxWithinType', 'LeftID', 'RightID', 'ChoiceKey', 'ChoiceSide', 'ChosenID', 'ChosenPwin', 'RT', 'TrialPoints', 'W1_Pwin', 'W2_Pwin', 'L1_Pwin', 'L2_Pwin']
    existing_cols = [c for c in cols_to_keep if c in df.columns]
        
    all_data.append(df[existing_cols])

print(f"\nExcluded {len(excluded_subs)} subjects due to reversal: {', '.join(excluded_subs)}")
print(f"Excluded {len(manually_removed)} subjects manually/incomplete: {', '.join(manually_removed)}")
print(f"Number of kept unique subs: {len(kept_subs_trials)}")

trial_dist = Counter(kept_subs_trials.values())
print("\nTrial count distribution (kept subs):")
for trials, count in trial_dist.items():
    print(f"  {count} subs had {trials} trials.")

if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    out_path = '/Users/nachshonkorem/Documents/AntiGrav/Anxiety/data/allsubs_day1.csv'
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    final_df.to_csv(out_path, index=False)
    print(f"\nSaved concatenated data to {out_path}")


Found 77 files matching the pattern.



Excluded 5 subjects due to reversal: 1, 3, 4, 5, 6
Excluded 5 subjects manually/incomplete: 26, 40, 50, 78, 79
Number of kept unique subs: 67

Trial count distribution (kept subs):
  67 subs had 100 trials.

Saved concatenated data to /Users/nachshonkorem/Documents/AntiGrav/Anxiety/data/allsubs_day1.csv
